In [1]:
# Import Code
from RTJ import RTJ_SCHEMA
from GD import GD_SCHEMA
import os
import torch
import librosa
import pandas as pd
import numpy as np
from muq import MuQMuLan
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Merge for a full analysis dictionary
full_catalog = {**RTJ_SCHEMA, **GD_SCHEMA}

In [2]:
# Import Model
device = 'mps'
mulan = MuQMuLan.from_pretrained("OpenMuQ/MuQ-MuLan-large")
mulan = mulan.to(device).eval()

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# BATCHED
def get_text_embedding(mulan, text, device, min_words=10):
    """
    Returns a text embedding from MuLan, handling token-limit errors by
    recursively splitting the text in half and averaging the embeddings.
    """
    try:
        with torch.no_grad():
            return mulan(texts=[text])
    except Exception as e:
        # Check if the error is token-length related
        error_msg = str(e).lower()
        if not any(kw in error_msg for kw in ["bounds"]):
            raise  # Re-raise if it's an unrelated error

        words = text.split()
        if len(words) < min_words * 2:
            # Text is too short to split further — just truncate to be safe
            print(f"  Warning: text too short to split further ({len(words)} words). Truncating.")
            truncated = " ".join(words[:len(words) // 2])
            with torch.no_grad():
                return mulan(texts=[truncated])

        # Split in half and recurse
        mid = len(words) // 2
        first_half = " ".join(words[:mid])
        second_half = " ".join(words[mid:])

        print(f"  Token limit hit — splitting text ({len(words)} words) into two halves and averaging.")
        emb_first = get_text_embedding(mulan, first_half, device, min_words)
        emb_second = get_text_embedding(mulan, second_half, device, min_words)

        # Average the two embeddings
        return (emb_first + emb_second) / 2


def get_audio_embedding(mulan, file_path, device):
    """Loads audio at 24kHz and returns the MuQ-MuLan embedding."""
    wav, _ = librosa.load(file_path, sr=24000)
    wavs = torch.tensor(wav).unsqueeze(0).to(device)
    with torch.no_grad():
        audio_embeds = mulan(wavs=wavs)
    return audio_embeds


def run_analysis(schema, folder_path, model_name="OpenMuQ/MuQ-MuLan-large"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model {model_name} on {device}...")
    mulan = MuQMuLan.from_pretrained(model_name).to(device).eval()

    ids = []
    text_embeddings = []
    audio_embeddings = []

    for file_id, data in schema.items():
        lyrics = data['lyrics']

        audio_file = None
        for ext in ['.wav', '.mp3', '.flac']:
            potential_path = os.path.join(folder_path, f"{file_id}{ext}")
            if os.path.exists(potential_path):
                audio_file = potential_path
                break

        if audio_file:
            print(f"Found and processing: {audio_file}")
            ids.append(file_id)

            # 1. Audio Embedding
            a_emb = get_audio_embedding(mulan, audio_file, device)
            audio_embeddings.append(a_emb)

            # 2. Text Embedding (with automatic batching if too long)
            print(f"  Generating text embedding for {file_id}...")
            t_emb = get_text_embedding(mulan, lyrics, device)
            text_embeddings.append(t_emb)
        else:
            print(f"Warning: File {file_id}.wav (or .mp3/.flac) not found in {folder_path}")

    if not audio_embeddings:
        raise ValueError(f"No audio files matching the IDs were found in the folder: {folder_path}")

    # 3. Concatenate and Calculate Similarities
    audio_tensor = torch.cat(audio_embeddings, dim=0)
    text_tensor = torch.cat(text_embeddings, dim=0)
    sim_matrix = mulan.calc_similarity(audio_tensor, text_tensor)

    # 4. Format into a DataFrame
    df = pd.DataFrame(
        sim_matrix.cpu().numpy(),
        index=[f"{fid} (Audio)" for fid in ids],
        columns=[f"{fid} (Lyrics)" for fid in ids]
    )
    return df

In [4]:
# Combined Analysis
def run_combined_analysis(sources, model_name="OpenMuQ/MuQ-MuLan-large", save_dir="results"):
    """
    sources:  list of (schema, folder_path) tuples
    save_dir: directory to write all outputs

    Saves:
      - results/similarity_matrix.csv
      - results/audio_embeddings.npz
      - results/text_embeddings.npz

    Returns:
      sim_df                — similarity DataFrame (N×N)
      audio_embeddings_dict — {id: np.ndarray}
      text_embeddings_dict  — {id: np.ndarray}
      id_to_title           — {id: song title string}
    """
    os.makedirs(save_dir, exist_ok=True)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading model {model_name} on {device}...")
    mulan = MuQMuLan.from_pretrained(model_name).to(device).eval()

    ids                   = []
    audio_embeddings_list = []
    text_embeddings_list  = []
    id_to_title           = {}

    for schema, folder_path in sources:
        for file_id, data in schema.items():
            lyrics = data["lyrics"]
            id_to_title[file_id] = data["title"]

            audio_file = None
            for ext in [".wav", ".mp3", ".flac"]:
                potential_path = os.path.join(folder_path, f"{file_id}{ext}")
                if os.path.exists(potential_path):
                    audio_file = potential_path
                    break

            if audio_file:
                print(f"Found and processing: {audio_file}")
                ids.append(file_id)

                print(f"  Generating audio embedding for {file_id}...")
                audio_embeddings_list.append(get_audio_embedding(mulan, audio_file, device))

                print(f"  Generating text embedding for {file_id}...")
                text_embeddings_list.append(get_text_embedding(mulan, lyrics, device))
            else:
                print(f"Warning: {file_id}.wav/.mp3/.flac not found in {folder_path}")

    if not audio_embeddings_list:
        raise ValueError("No audio files were found across any of the provided sources.")

    audio_tensor = torch.cat(audio_embeddings_list, dim=0)
    text_tensor  = torch.cat(text_embeddings_list,  dim=0)

    sim_matrix = mulan.calc_similarity(audio_tensor, text_tensor)
    sim_df = pd.DataFrame(
        sim_matrix.cpu().numpy(),
        index=[f"{fid} (Audio)"  for fid in ids],
        columns=[f"{fid} (Lyrics)" for fid in ids],
    )
    sim_df.to_csv(os.path.join(save_dir, "similarity_matrix.csv"))
    print(f"\nSaved similarity matrix → {save_dir}/similarity_matrix.csv")

    audio_np = audio_tensor.cpu().numpy()
    text_np  = text_tensor.cpu().numpy()

    audio_embeddings_dict = {fid: audio_np[i] for i, fid in enumerate(ids)}
    text_embeddings_dict  = {fid: text_np[i]  for i, fid in enumerate(ids)}

    np.savez(os.path.join(save_dir, "audio_embeddings.npz"), **audio_embeddings_dict)
    np.savez(os.path.join(save_dir, "text_embeddings.npz"),  **text_embeddings_dict)
    print(f"Saved audio embeddings → {save_dir}/audio_embeddings.npz")
    print(f"Saved text embeddings  → {save_dir}/text_embeddings.npz")

    return sim_df, audio_embeddings_dict, text_embeddings_dict, id_to_title

In [5]:
def _make_label(file_id, data_type, id_to_title):
    """
    Formats a point label: '{corpus} {S|L} {Song Title}'
    e.g. 'RTJ L Banana Clipper' or 'GD S Shakedown Street'
    """
    corpus = "RTJ" if file_id.startswith("RTJ") else "GD"
    kind   = "S" if data_type == "audio" else "L"
    title  = id_to_title.get(file_id, file_id)
    return f"{corpus} {kind} {title}"


def run_pca(audio_embeddings_dict, text_embeddings_dict, id_to_title,
            n_components=10, save_dir="results"):
    """
    Runs PCA four ways:
      1. audio    — audio embeddings only       (N points, D dims)
      2. text     — text embeddings only        (N points, D dims)
      3. joint    — audio+text concatenated     (N points, 2D dims)
      4. combined — all audio and text stacked  (2N points, D dims)

    The 'combined' run is used for the all-40-points plot.

    Returns a dict of result dicts keyed by run name.
    """
    os.makedirs(save_dir, exist_ok=True)

    ids      = list(audio_embeddings_dict.keys())
    audio_np = np.stack([audio_embeddings_dict[fid] for fid in ids])  # [N, D]
    text_np  = np.stack([text_embeddings_dict[fid]  for fid in ids])  # [N, D]
    joint_np = np.concatenate([audio_np, text_np], axis=1)            # [N, 2D]

    # --- Runs 1–3: one row per track ---
    standard_runs = {
        "audio": (audio_np, ids, ["audio"] * len(ids)),
        "text":  (text_np,  ids, ["text"]  * len(ids)),
        "joint": (joint_np, ids, ["joint"] * len(ids)),
    }

    # --- Run 4: one row per (track × modality) ---
    combined_ids   = ids + ids
    combined_types = ["audio"] * len(ids) + ["text"] * len(ids)
    combined_np    = np.vstack([audio_np, text_np])                    # [2N, D]

    all_runs = {**standard_runs, "combined": (combined_np, combined_ids, combined_types)}

    results = {}

    for run_name, (matrix, row_ids, row_types) in all_runs.items():
        print(f"\nRunning PCA on {run_name} embeddings (shape {matrix.shape})...")

        scaler        = StandardScaler()
        matrix_scaled = scaler.fit_transform(matrix)

        n_comp      = min(n_components, matrix_scaled.shape[0], matrix_scaled.shape[1])
        pca         = PCA(n_components=n_comp)
        projections = pca.fit_transform(matrix_scaled)

        ev_df = pd.DataFrame({
            "PC":                  [f"PC{i+1}" for i in range(n_comp)],
            "explained_variance":  pca.explained_variance_ratio_,
            "cumulative_variance": np.cumsum(pca.explained_variance_ratio_),
        })

        proj_df = pd.DataFrame(
            projections,
            columns=[f"PC{i+1}" for i in range(n_comp)],
        )
        proj_df.insert(0, "file_id", row_ids)
        proj_df.insert(1, "corpus", ["RTJ" if fid.startswith("RTJ") else "GD" for fid in row_ids])
        proj_df.insert(2, "data_type", row_types)
        proj_df.insert(3, "label", [_make_label(fid, dt, id_to_title)
                                    for fid, dt in zip(row_ids, row_types)])

        ev_df.to_csv(  os.path.join(save_dir, f"pca_{run_name}_components.csv"),  index=False)
        proj_df.to_csv(os.path.join(save_dir, f"pca_{run_name}_projections.csv"), index=False)

        if n_comp >= 3:
            print(f"  Top-3 PCs explain {ev_df['cumulative_variance'].iloc[2]:.1%} of variance")
        print(f"  Saved → {save_dir}/pca_{run_name}_components.csv")
        print(f"  Saved → {save_dir}/pca_{run_name}_projections.csv")

        results[run_name] = {"projections": proj_df, "explained_variance": ev_df, "pca": pca}

    return results


def plot_pca(pca_results, save_dir="results"):
    """
    For runs audio / text / joint: scatterplot coloured by corpus, labelled by title.
    For the combined run: colour = corpus (RTJ vs GD), shape = data type (audio vs text).
    """
    os.makedirs(save_dir, exist_ok=True)

    corpus_colors = {"RTJ": "#E63946", "GD": "#457B9D"}
    type_markers  = {"audio": "o", "text": "^"}   # circle = song, triangle = lyrics

    for run_name, result in pca_results.items():
        proj_df = result["projections"]
        ev_df   = result["explained_variance"]

        pc1_var = ev_df.loc[ev_df["PC"] == "PC1", "explained_variance"].values[0]
        pc2_var = ev_df.loc[ev_df["PC"] == "PC2", "explained_variance"].values[0]

        fig, ax = plt.subplots(figsize=(10, 7))

        if run_name == "combined":
            # Colour by corpus, shape by data type
            for (corpus, dtype), group in proj_df.groupby(["corpus", "data_type"]):
                ax.scatter(
                    group["PC1"], group["PC2"],
                    label=f"{corpus} — {dtype}",
                    color=corpus_colors.get(corpus, "grey"),
                    marker=type_markers.get(dtype, "o"),
                    s=90, alpha=0.85, edgecolors="white", linewidths=0.5,
                )
            # Legend: two corpus colour patches + two shape entries
            from matplotlib.lines import Line2D
            legend_elements = [
                Line2D([0], [0], marker="o", color="w", markerfacecolor=corpus_colors["RTJ"],
                       markersize=9, label="RTJ"),
                Line2D([0], [0], marker="o", color="w", markerfacecolor=corpus_colors["GD"],
                       markersize=9, label="GD"),
                Line2D([0], [0], marker="o", color="grey", linestyle="None",
                       markersize=9, label="Song (audio)"),
                Line2D([0], [0], marker="^", color="grey", linestyle="None",
                       markersize=9, label="Lyrics (text)"),
            ]
            ax.legend(handles=legend_elements, loc="best")
            ax.set_title("Comparing Semantic Similarity of Songs and Lyrics for Grateful Dead and Run the Jewels")

        else:
            # Colour by corpus only
            for corpus, group in proj_df.groupby("corpus"):
                ax.scatter(
                    group["PC1"], group["PC2"],
                    label=corpus,
                    color=corpus_colors.get(corpus, "grey"),
                    s=80, alpha=0.85, edgecolors="white", linewidths=0.5,
                )
            ax.legend()
            ax.set_title(f"PCA — {run_name} embeddings")

        # Annotate every point with its formatted label
        for _, row in proj_df.iterrows():
            ax.annotate(
                row["label"],
                (row["PC1"], row["PC2"]),
                fontsize=4.5, alpha=0.75,
                xytext=(4, 4), textcoords="offset points",
            )

        ax.set_xlabel(f"Dimension 1 ({pc1_var:.1%} variance)")
        ax.set_ylabel(f"Dimension 2 ({pc2_var:.1%} variance)")
        fig.tight_layout()

        path = os.path.join(save_dir, f"pca_{run_name}.png")
        fig.savefig(path, dpi=150)
        plt.close(fig)
        print(f"Saved plot → {path}")

In [6]:
# --- EXECUTION ---
sim_df, audio_embs, text_embs, id_to_title = run_combined_analysis(
    sources=[(RTJ_SCHEMA, "RTJ"), (GD_SCHEMA, "GD")],
    save_dir="results",
)

Loading model OpenMuQ/MuQ-MuLan-large on cpu...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Found and processing: RTJ/RTJ_01.mp3
  Generating audio embedding for RTJ_01...
  Generating text embedding for RTJ_01...


Token indices sequence length is longer than the specified maximum sequence length for this model (919 > 512). Running this sequence through the model will result in indexing errors


  Token limit hit — splitting text (555 words) into two halves and averaging.
Found and processing: RTJ/RTJ_02.mp3
  Generating audio embedding for RTJ_02...
  Generating text embedding for RTJ_02...
  Token limit hit — splitting text (579 words) into two halves and averaging.
Found and processing: RTJ/RTJ_03.mp3
  Generating audio embedding for RTJ_03...
  Generating text embedding for RTJ_03...
  Token limit hit — splitting text (520 words) into two halves and averaging.
Found and processing: RTJ/RTJ_04.mp3
  Generating audio embedding for RTJ_04...
  Generating text embedding for RTJ_04...
  Token limit hit — splitting text (368 words) into two halves and averaging.
Found and processing: RTJ/RTJ_05.mp3
  Generating audio embedding for RTJ_05...
  Generating text embedding for RTJ_05...
  Token limit hit — splitting text (627 words) into two halves and averaging.
Found and processing: RTJ/RTJ_06.mp3
  Generating audio embedding for RTJ_06...
  Generating text embedding for RTJ_06...


In [7]:
# PCA
pca_results = run_pca(audio_embs, text_embs, id_to_title, n_components=10, save_dir="results")
plot_pca(pca_results, save_dir="results")


Running PCA on audio embeddings (shape (20, 512))...
  Top-3 PCs explain 73.5% of variance
  Saved → results/pca_audio_components.csv
  Saved → results/pca_audio_projections.csv

Running PCA on text embeddings (shape (20, 512))...
  Top-3 PCs explain 64.9% of variance
  Saved → results/pca_text_components.csv
  Saved → results/pca_text_projections.csv

Running PCA on joint embeddings (shape (20, 1024))...
  Top-3 PCs explain 60.6% of variance
  Saved → results/pca_joint_components.csv
  Saved → results/pca_joint_projections.csv

Running PCA on combined embeddings (shape (40, 512))...
  Top-3 PCs explain 73.2% of variance
  Saved → results/pca_combined_components.csv
  Saved → results/pca_combined_projections.csv
Saved plot → results/pca_audio.png
Saved plot → results/pca_text.png
Saved plot → results/pca_joint.png
Saved plot → results/pca_combined.png
